In [ ]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup, BertModel, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from tqdm import tqdm
import logging
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import LongformerForSequenceClassification
from transformers import LongformerTokenizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from transformers import LongformerModel, LongformerConfig
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.nn.utils.rnn import pad_sequence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        """
        A custom dataset for handling short texts with optional global attention.

        Args:
        - texts: List of input texts.
        - labels: List of corresponding labels.
        - tokenizer: The tokenizer to convert texts to token ids.
        - max_length: Maximum length for token sequences (used when dynamic_padding=False).
        - global_attention_target: Which tokens to assign global attention to. Options: 'cls', 'question_mark', 'custom'.
        - dynamic_padding: Whether to use dynamic padding based on the longest sequence in the batch.
        - use_global_attention: Boolean flag to enable or disable the global attention mask.
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Dynamic or fixed padding
        padding_strategy = "longest" if self.dynamic_padding else "max_length"

        # Tokenize the text with the appropriate padding and truncation
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        # Initialize the output dictionary
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)  # Convert labels to float for BCEWithLogitsLoss
        }

        # Add global attention mask if use_global_attention is True
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)

            # Apply global attention based on the chosen strategy
            if self.global_attention_target == 'cls':
                # Apply global attention to the CLS token (first token)
                global_attention_mask[0] = 1
            elif self.global_attention_target == 'question_mark':
                # Apply global attention to any question marks in the text
                question_mark_token_id = self.tokenizer.convert_tokens_to_ids("?")
                question_mark_position = (input_ids == question_mark_token_id).nonzero(as_tuple=True)
                if question_mark_position[0].numel() > 0:  # If there's a question mark
                    global_attention_mask[question_mark_position[0]] = 1
            elif self.global_attention_target == False:
                # Disable global attention
                global_attention_mask = torch.zeros_like(input_ids)
            elif self.global_attention_target == 'custom':
                # Implement custom logic to apply global attention to specific tokens
                pass  # Add custom logic here

            # Add global_attention_mask to the output dictionary
            output['global_attention_mask'] = global_attention_mask

        return output


In [ ]:
# Train function with BCEWithLogitsLoss
def train_one_epoch(model, data_loader, criterion, optimizer, device, scheduler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Initialize the progress bar
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc="Training")

    for batch_idx, data in pbar:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        labels = data['labels'].float().to(device)  # Convert labels to float for BCEWithLogitsLoss

        # Check if global_attention_mask is present in the batch
        global_attention_mask = data.get('global_attention_mask', None)
        if global_attention_mask is not None:
            global_attention_mask = global_attention_mask.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

        # Compute the loss (BCEWithLogitsLoss expects logits and float labels)
        loss = criterion(outputs.logits.view(-1), labels.float().view(-1))

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * input_ids.size(0)

        # Sigmoid to convert logits to probabilities
        probabilities = torch.sigmoid(outputs.logits.squeeze())

        # Predictions (>= 0.5 is classified as class 1)
        predicted = (probabilities >= 0.5).float()

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update the progress bar
        pbar.set_postfix({
            'loss': running_loss / ((batch_idx + 1) * data_loader.batch_size),
            'accuracy': 100 * correct / total
        })

    epoch_loss = running_loss / len(data_loader.dataset)
    epoch_accuracy = 100 * correct / total
    return epoch_loss, epoch_accuracy


In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # Initialize the progress bar
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc="Evaluating")

    with torch.no_grad():
        for batch_idx, data in pbar:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            labels = data['labels'].float().to(device)  # Ensure labels are in float

            # Check if global_attention_mask is present in the batch
            global_attention_mask = data.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

            # Use view(-1) instead of squeeze() to avoid removing batch dimension when it's 1
            loss = criterion(outputs.logits.view(-1), labels.view(-1))  # Ensure matching shapes for loss calculation
            running_loss += loss.item() * input_ids.size(0)

            # Sigmoid to convert logits to probabilities
            probabilities = torch.sigmoid(outputs.logits.view(-1))

            # Predictions (>= 0.5 is classified as class 1)
            predicted = (probabilities >= 0.5).float()

            correct += (predicted == labels.view(-1)).sum().item()
            total += labels.size(0)

            # Update the progress bar
            pbar.set_postfix({
                'loss': running_loss / ((batch_idx + 1) * data_loader.batch_size),
                'accuracy': 100 * correct / total
            })

    epoch_loss = running_loss / len(data_loader.dataset)
    epoch_accuracy = 100 * correct / total
    return epoch_loss, epoch_accuracy


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate_fn(batch):
    """
    Custom collate function to dynamically pad sequences within a batch to match the longest sequence.
    """
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    global_attention_mask = [item['global_attention_mask'] for item in batch]
    labels = torch.tensor([item['labels'] for item in batch], dtype=torch.float)  # Convert labels to float for BCEWithLogitsLoss

    # Pad the sequences in the batch to match the longest sequence in the batch
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    global_attention_mask = pad_sequence(global_attention_mask, batch_first=True, padding_value=0)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'global_attention_mask': global_attention_mask,
        'labels': labels  # Labels now in float format
    }


In [ ]:
def get_predictions(model, data_loader, device):
    """Gets predictions from the model on a given data loader."""
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Check if global_attention_mask is present in the batch
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

            # Apply sigmoid to get probabilities
            probabilities = torch.sigmoid(outputs.logits)

            # Convert probabilities to binary predictions (threshold = 0.5)
            predicted = (probabilities >= 0.5).float()

            # Ensure predictions and labels are 1-dimensional arrays before extending
            predictions.extend(predicted.view(-1).cpu().numpy())  # Flatten to 1D
            true_labels.extend(labels.view(-1).cpu().numpy())     # Flatten to 1D

    return np.array(predictions), np.array(true_labels)


In [ ]:
mimic_path = '/content/drive/My Drive/EHR_PROJ/MODELS/mimic_no_preprocess_no_glob_binary'


# Initialize the Longformer model for classification
mimic_model = LongformerForSequenceClassification.from_pretrained(mimic_path)
mimic_tokenizer = LongformerTokenizer.from_pretrained(mimic_path)

# Move the model to the appropriate device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mimic_model.to(device)

LongformerForSequenceClassification(
  (longformer): LongformerModel(
    (embeddings): LongformerEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(4098, 768, padding_idx=1)
    )
    (encoder): LongformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x LongformerLayer(
          (attention): LongformerAttention(
            (self): LongformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (query_global): Linear(in_features=768, out_features=768, bias=True)
              (key_global): Linear(in_features=768, out_features=768, bias=True)
          

In [ ]:
berkeley_mimic_path = '/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_mimic_v010725'

# Initialize the Longformer model for classification
berkeley_mimic_model = LongformerForSequenceClassification.from_pretrained(berkeley_mimic_path)
berkeley_mimic_tokenizer = LongformerTokenizer.from_pretrained(berkeley_mimic_path)

berkeley_mimic_model.to(device)

LongformerForSequenceClassification(
  (longformer): LongformerModel(
    (embeddings): LongformerEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(4098, 768, padding_idx=1)
    )
    (encoder): LongformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x LongformerLayer(
          (attention): LongformerAttention(
            (self): LongformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (query_global): Linear(in_features=768, out_features=768, bias=True)
              (key_global): Linear(in_features=768, out_features=768, bias=True)
          

In [ ]:
berkeley_phenotype_mimic_path = '/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_phenotype_to_mimic_v030325'

# Initialize the Longformer model for classification
berkeley_phenotype_mimic_model = LongformerForSequenceClassification.from_pretrained(berkeley_phenotype_mimic_path)
berkeley_phenotype_mimic_tokenizer = LongformerTokenizer.from_pretrained(berkeley_phenotype_mimic_path)

berkeley_phenotype_mimic_model.to(device)

LongformerForSequenceClassification(
  (longformer): LongformerModel(
    (embeddings): LongformerEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(4098, 768, padding_idx=1)
    )
    (encoder): LongformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x LongformerLayer(
          (attention): LongformerAttention(
            (self): LongformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (query_global): Linear(in_features=768, out_features=768, bias=True)
              (key_global): Linear(in_features=768, out_features=768, bias=True)
          

In [ ]:
# 0.727603
# mimic_train = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_train.csv')
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')

In [ ]:
# train_texts = mimic_train['text'].to_numpy()
# train_labels = mimic_train['label'].to_numpy()
val_texts = mimic_test['text'].to_numpy()
val_labels = mimic_test['label'].to_numpy()

In [ ]:
# Create 35 random subsets from val_texts and val_labels, each using 70% of the original data. Then, the models and tokenizers created above will be used to predict the subsets. Record each model's accuracy, f-1, precision, recall, and AUC.

results = []
for i in range(35):
    # Create a random subset of 70% of the data
    subset_indices = np.random.choice(len(val_texts), size=int(0.7 * len(val_texts)), replace=False)
    subset_texts = val_texts[subset_indices]
    subset_labels = val_labels[subset_indices]

    # Create datasets and data loaders for the subset
    subset_dataset_mimic = TextDataset(subset_texts, subset_labels, mimic_tokenizer, use_global_attention=False)
    subset_loader_mimic = DataLoader(subset_dataset_mimic, batch_size=32)

    subset_dataset_berkeley = TextDataset(subset_texts, subset_labels, berkeley_mimic_tokenizer, use_global_attention=False)
    subset_loader_berkeley = DataLoader(subset_dataset_berkeley, batch_size=32)

    subset_dataset_berkeley_phenotype = TextDataset(subset_texts, subset_labels, berkeley_phenotype_mimic_tokenizer, use_global_attention=False)
    subset_loader_berkeley_phenotype = DataLoader(subset_dataset_berkeley_phenotype, batch_size=32)

    # Get predictions for each model
    mimic_preds, mimic_true = get_predictions(mimic_model, subset_loader_mimic, device)
    berkeley_preds, berkeley_true = get_predictions(berkeley_mimic_model, subset_loader_berkeley, device)
    berkeley_phenotype_preds, berkeley_phenotype_true = get_predictions(berkeley_phenotype_mimic_model, subset_loader_berkeley_phenotype, device)

    # Calculate metrics for each model
    for model_name, preds, true in zip(["mimic", "berkeley_mimic", "berkeley_phenotype_mimic"], [mimic_preds, berkeley_preds, berkeley_phenotype_preds], [mimic_true, berkeley_true, berkeley_phenotype_true]):
        accuracy = accuracy_score(true, preds)
        f1 = f1_score(true, preds)
        precision = precision_score(true, preds)
        recall = recall_score(true, preds)
        auc = roc_auc_score(true, preds)

        results.append({
            "subset": i,
            "model": model_name,
            "accuracy": accuracy,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc
        })

Initializing global attention on CLS token...


In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df.head(30)

,subset,model,accuracy,f1,precision,recall,auc
0,0,mimic,0.861592,0.907193,0.900922,0.913551,0.813442
1,0,berkeley_mimic,0.885813,0.924829,0.902222,0.948598,0.827632
2,0,berkeley_phenotype_mimic,0.903114,0.936219,0.913333,0.960280,0.850140
3,1,mimic,0.847751,0.895238,0.884706,0.906024,0.802705
4,1,berkeley_mimic,0.875433,0.915294,0.894253,0.937349,0.827570
5,1,berkeley_phenotype_mimic,0.891003,0.926316,0.900000,0.954217,0.842139
6,2,mimic,0.849481,0.898007,0.882488,0.914081,0.796663
7,2,berkeley_mimic,0.878893,0.918605,0.895692,0.942721,0.826706
8,2,berkeley_phenotype_mimic,0.899654,0.932558,0.909297,0.957041,0.852734
9,3,mimic,0.865052,0.909722,0.893182,0.926887,0.810846


In [ ]:
# Create separate DataFrames for each model
mimic_df = results_df[results_df["model"] == "mimic"].reset_index(drop=True)
berkeley_mimic_df = results_df[results_df["model"] == "berkeley_mimic"].reset_index(drop=True)
berkeley_phenotype_mimic_df = results_df[results_df["model"] == "berkeley_phenotype_mimic"].reset_index(drop=True)

In [ ]:
mimic_df.describe()

,subset,accuracy,f1,precision,recall,auc
count,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000
mean,17.000000,0.860900,0.905699,0.895064,0.916658,0.813842
std,10.246951,0.008052,0.005954,0.009472,0.006521,0.011222
min,0.000000,0.846021,0.895238,0.875878,0.903002,0.791672
25%,8.500000,0.853806,0.900700,0.889412,0.911374,0.804105
50%,17.000000,0.861592,0.906141,0.893519,0.918415,0.813954
75%,25.500000,0.865052,0.909513,0.901597,0.920893,0.820818
max,34.000000,0.873702,0.915802,0.912530,0.928741,0.838756


In [ ]:
berkeley_mimic_df.describe()

,subset,accuracy,f1,precision,recall,auc
count,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000
mean,17.000000,0.879881,0.919469,0.898952,0.941020,0.828336
std,10.246951,0.007356,0.005317,0.009020,0.006394,0.010177
min,0.000000,0.863322,0.907386,0.878049,0.923788,0.804013
25%,8.500000,0.875433,0.916522,0.894554,0.937722,0.822347
50%,17.000000,0.878893,0.919540,0.899543,0.940758,0.828036
75%,25.500000,0.884083,0.922002,0.902715,0.945818,0.834833
max,34.000000,0.896194,0.931818,0.922197,0.951276,0.855603


In [ ]:
berkeley_phenotype_mimic_df.describe()

,subset,accuracy,f1,precision,recall,auc
count,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000
mean,17.000000,0.897776,0.931510,0.910076,0.954032,0.850331
std,10.246951,0.006158,0.004555,0.007789,0.005204,0.008233
min,0.000000,0.884083,0.921615,0.889908,0.936321,0.830056
25%,8.500000,0.893599,0.928904,0.907111,0.952038,0.846015
50%,17.000000,0.897924,0.931634,0.910550,0.954217,0.851303
75%,25.500000,0.902249,0.935095,0.913043,0.957143,0.855545
max,34.000000,0.908304,0.939567,0.927273,0.961905,0.868142


In [ ]:
# Save the DataFrames to CSV files in your Google Drive
mimic_df.to_csv('/content/drive/My Drive/EHR_PROJ/Results/mimic_df_2.csv', index=False)
berkeley_mimic_df.to_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_mimic_df_2.csv', index=False)
berkeley_phenotype_mimic_df.to_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_phenotype_mimic_df_2.csv', index=False)

In [ ]:
# prompt: do z-test to compare accuracy, f1, precision, recall, auc from mimic_df, berkeley_mimic_df, and berkeley_phenotype_mimic_df

import pandas as pd
from scipy import stats

# Load the DataFrames from your Google Drive
mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/mimic_df_2.csv')
berkeley_mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_mimic_df_2.csv')
berkeley_phenotype_mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_phenotype_mimic_df_2.csv')

# Define a function to perform the z-test and print the results
def perform_ztest(metric, df1, df2, df1_name, df2_name):
    z_stat, p_val = stats.ttest_ind(df1[metric], df2[metric]) # Use t-test since sample size is less than 30
    print(f"Z-test for {metric} between {df1_name} and {df2_name}:")
    print(f"  z-statistic: {z_stat:.4f}")
    print(f"  p-value: {p_val:.4f}")
    print("-" * 30)

# Perform z-tests for each metric and model comparison
metrics = ['accuracy', 'f1', 'precision', 'recall', 'auc']

for metric in metrics:
    perform_ztest(metric, mimic_df, berkeley_mimic_df, "mimic_df", "berkeley_mimic_df")
    perform_ztest(metric, mimic_df, berkeley_phenotype_mimic_df, "mimic_df", "berkeley_phenotype_mimic_df")
    perform_ztest(metric, berkeley_mimic_df, berkeley_phenotype_mimic_df, "berkeley_mimic_df", "berkeley_phenotype_mimic_df")

Z-test for accuracy between mimic_df and berkeley_mimic_df:
  z-statistic: -10.2971
  p-value: 0.0000
------------------------------
Z-test for accuracy between mimic_df and berkeley_phenotype_mimic_df:
  z-statistic: -21.5220
  p-value: 0.0000
------------------------------
Z-test for accuracy between berkeley_mimic_df and berkeley_phenotype_mimic_df:
  z-statistic: -11.0355
  p-value: 0.0000
------------------------------
Z-test for f1 between mimic_df and berkeley_mimic_df:
  z-statistic: -10.2056
  p-value: 0.0000
------------------------------
Z-test for f1 between mimic_df and berkeley_phenotype_mimic_df:
  z-statistic: -20.3685
  p-value: 0.0000
------------------------------
Z-test for f1 between berkeley_mimic_df and berkeley_phenotype_mimic_df:
  z-statistic: -10.1750
  p-value: 0.0000
------------------------------
Z-test for precision between mimic_df and berkeley_mimic_df:
  z-statistic: -1.7587
  p-value: 0.0831
------------------------------
Z-test for precision between 

In [ ]:
# prompt: modify the upper block to conduct T test

import pandas as pd
from scipy import stats

# ... (Your existing code) ...

# Load the DataFrames from your Google Drive
mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/mimic_df_2.csv')
berkeley_mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_mimic_df_2.csv')
berkeley_phenotype_mimic_df = pd.read_csv('/content/drive/My Drive/EHR_PROJ/Results/berkeley_phenotype_mimic_df_2.csv')

# Define a function to perform the t-test and print the results
def perform_ttest(metric, df1, df2, df1_name, df2_name):
    t_stat, p_val = stats.ttest_ind(df1[metric], df2[metric])
    print(f"T-test for {metric} between {df1_name} and {df2_name}:")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_val:.4f}")
    print("-" * 30)

# Perform t-tests for each metric and model comparison
metrics = ['accuracy', 'f1', 'precision', 'recall', 'auc']

for metric in metrics:
    perform_ttest(metric, mimic_df, berkeley_mimic_df, "mimic_df", "berkeley_mimic_df")
    perform_ttest(metric, mimic_df, berkeley_phenotype_mimic_df, "mimic_df", "berkeley_phenotype_mimic_df")
    perform_ttest(metric, berkeley_mimic_df, berkeley_phenotype_mimic_df, "berkeley_mimic_df", "berkeley_phenotype_mimic_df")


T-test for accuracy between mimic_df and berkeley_mimic_df:
  t-statistic: -10.2971
  p-value: 0.0000
------------------------------
T-test for accuracy between mimic_df and berkeley_phenotype_mimic_df:
  t-statistic: -21.5220
  p-value: 0.0000
------------------------------
T-test for accuracy between berkeley_mimic_df and berkeley_phenotype_mimic_df:
  t-statistic: -11.0355
  p-value: 0.0000
------------------------------
T-test for f1 between mimic_df and berkeley_mimic_df:
  t-statistic: -10.2056
  p-value: 0.0000
------------------------------
T-test for f1 between mimic_df and berkeley_phenotype_mimic_df:
  t-statistic: -20.3685
  p-value: 0.0000
------------------------------
T-test for f1 between berkeley_mimic_df and berkeley_phenotype_mimic_df:
  t-statistic: -10.1750
  p-value: 0.0000
------------------------------
T-test for precision between mimic_df and berkeley_mimic_df:
  t-statistic: -1.7587
  p-value: 0.0831
------------------------------
T-test for precision between 